# SMART - Session Matching And Automated Recommendation Tool

This notebook demonstrates the complete SMART workflow for organizing conference sessions based on abstract similarity using text embeddings and LLMs.

## Workflow Overview
1. **Configuration** - Set file paths, API keys, and parameters
2. **Load Data** - Import presentations and hybrid sessions
3. **Generate Embeddings** - Create text embeddings using Gemini
4. **Remove Duplicates** - Identify and remove near-duplicate submissions
5. **Create Sessions** - Cluster presentations into sessions
6. **Calculate Metrics** - Compute coherence and distinctiveness
7. **Generate Titles** - Use LLM to create session titles and keywords
8. **Match Committees** - Assign committees based on topic similarity
9. **Export Results** - Save outputs for review and publication

---
## 1. Configuration

Set all configuration options here. Modify these values to match your data files and preferences.

In [1]:
# ============================================================
# CONFIGURATION - Modify these values for your conference
# ============================================================

# Conference settings
CONFERENCE_YEAR = 26  # Two-digit year (e.g., 26 for 2026)
CONFERENCE_NAME = "AIM2026-TestNB"

# Input files
PRESENTATIONS_FILE = "abstracts 1.20.26.xlsx"  # Main presentations file
HYBRID_SESSIONS_FILE = "Hybrid Session Invited Presentations.xlsx"  # Pre-assigned sessions (optional)
COMMITTEES_FILE = "ASABE Committees.csv"  # Committee data for matching (optional)

# Column mappings for main presentations file
PRESENTATIONS_COLUMNS = {
    "title": "Submission-Call for Abstracts-Presentation Title-Character max 160",
    "abstract": "Submission-Call for Abstracts-Abstract-Character max 4000-Abstracts will only be used to group into topical sessions and evaluate quality of talk.",
    "submission_id": "Submission-Call for Abstracts-Submission ID - 7 digits",
    "session_preference": "Submission-Call for Abstracts-Select Your Session Preference",
    "technical_community": "Submission-Call for Abstracts-Technical Community-First Preference ",
    "presenter_first_name": "Owner-First Name",
    "presenter_last_name": "Owner-Last Name",
    "presenter_email": "Owner-E-mail Address",
}

# Column mappings for hybrid sessions file
HYBRID_COLUMNS = {
    "title": "Title",
    "abstract": "Abstract",
    "submission_id": "Submission ID - 7 digits",
    "session": "Session",
    "technical_community": "Technical Community",
    "presenter_first_name": "Presenter: First Name",
    "presenter_last_name": "Presenter: Last Name",
}

# Session filter (for oral vs poster)
ORAL_PREFERENCE_TEXT = "Oral. I would like this submission to be considered for an Oral (standard or lightning) session."

# Session creation parameters
MIN_SESSION_SIZE = 9
MAX_SESSION_SIZE = 12
MAX_SESSIONS = 111

# Embedding model
EMBEDDING_MODEL = "gemini-embedding-001"

# Title generation model
TITLE_MODEL = "gemini-2.5-flash-lite"

# Duplicate detection threshold (0.99 = very strict)
DUPLICATE_THRESHOLD = 0.99

# Database files (for caching)
EMBEDDING_CACHE_DB = f"{CONFERENCE_NAME}_embeddings.db"
CONFERENCE_DB = f"{CONFERENCE_NAME}_working.db"

# Output files
OUTPUT_DIR = "output"

print("Configuration loaded successfully!")
print(f"Conference: {CONFERENCE_NAME}")
print(f"Presentations file: {PRESENTATIONS_FILE}")
print(f"Hybrid sessions file: {HYBRID_SESSIONS_FILE}")

Configuration loaded successfully!
Conference: AIM2026-TestNB
Presentations file: abstracts 1.20.26.xlsx
Hybrid sessions file: Hybrid Session Invited Presentations.xlsx


---
## 2. Setup and Imports

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# Load SMART library
from smart import (
    EmbeddingCache, ConferenceDB, EmbeddingConfig,
    SessionConstraints, PlacementResult,
    create_placement_strategy,
    calculate_all_metrics, find_outlier_presentations,
)
from smart.io.loaders import (
    load_presentations, load_hybrid_sessions, load_committees,
    ColumnMapping, inspect_file,
)
from smart.llm.embeddings import GeminiEmbedder, CachedEmbedder
from smart.llm.titles import GeminiTitleGenerator

# Load environment variables (API keys, etc.)
load_dotenv(".env")

# Verify API key is available
if "GEMINI_API_KEY" not in os.environ:
    raise ValueError("GEMINI_API_KEY not found. Please set it in your .env file.")

# Create output directory
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print("✓ Libraries imported successfully")
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Libraries imported successfully
✓ Output directory: output


---
## 3. Initialize Databases and Embedder

In [3]:
# Initialize embedding cache (stores embeddings to avoid recomputation)
embedding_cache = EmbeddingCache(EMBEDDING_CACHE_DB)
print(f"✓ Embedding cache initialized: {EMBEDDING_CACHE_DB}")

# Initialize conference database (stores presentations, sessions, etc.)
conference_db = ConferenceDB(CONFERENCE_DB, conference_year=CONFERENCE_YEAR)
print(f"✓ Conference database initialized: {CONFERENCE_DB}")

# Initialize embedder with caching
gemini_embedder = GeminiEmbedder(model=EMBEDDING_MODEL)
embedder = CachedEmbedder(gemini_embedder, embedding_cache)
print(f"✓ Embedder initialized: {EMBEDDING_MODEL}")

# Show cache stats
cache_stats = embedding_cache.get_stats()
print(f"\nCache contains {cache_stats['total_embeddings']} embeddings")

✓ Embedding cache initialized: AIM2026-TestNB_embeddings.db
✓ Conference database initialized: AIM2026-TestNB_working.db
✓ Embedder initialized: gemini-embedding-001

Cache contains 0 embeddings


---
## 4. Load Hybrid Sessions (Invited Presentations)

Load pre-assigned hybrid sessions with invited presentations. These sessions are already organized and will be used as "anchors" for clustering.

In [4]:
# Inspect hybrid sessions file structure (optional)
if Path(HYBRID_SESSIONS_FILE).exists():
    info = inspect_file(HYBRID_SESSIONS_FILE)
    print(f"Hybrid file: {info['row_count']} rows, {info['column_count']} columns")
    print("\nColumns:")
    for col in info['columns'][:10]:
        print(f"  - {col['name']} ({col['dtype']})")
else:
    print(f"Hybrid file not found: {HYBRID_SESSIONS_FILE}")
    print("Skipping hybrid sessions...")

Hybrid file: 34 rows, 8 columns

Columns:
  - Session ID (int64)
  - Session (object)
  - Title (object)
  - Abstract (object)
  - Submission ID - 7 digits (float64)
  - Technical Community (object)
  - Presenter: First Name (object)
  - Presenter: Last Name (object)


In [5]:
# Load hybrid sessions
df_hybrid = None
df_hybrid_sessions = None
hybrid_metadata = None

if Path(HYBRID_SESSIONS_FILE).exists():
    # Create column mapping for hybrid file
    hybrid_mapping = ColumnMapping(
        title=HYBRID_COLUMNS["title"],
        abstract=HYBRID_COLUMNS["abstract"],
        submission_id=HYBRID_COLUMNS["submission_id"],
        session=HYBRID_COLUMNS["session"],
        technical_community=HYBRID_COLUMNS.get("technical_community"),
        presenter_first_name=HYBRID_COLUMNS.get("presenter_first_name"),
        presenter_last_name=HYBRID_COLUMNS.get("presenter_last_name"),
    )
    
    # Load hybrid sessions
    df_hybrid, df_hybrid_sessions, hybrid_metadata = load_hybrid_sessions(
        HYBRID_SESSIONS_FILE,
        hybrid_mapping,
        conference_year=CONFERENCE_YEAR,
    )
    
    print(f"✓ Loaded {len(df_hybrid)} hybrid presentation slots")
    print(f"✓ Found {hybrid_metadata['session_count']} hybrid sessions")
    print(f"✓ Unique presentations: {hybrid_metadata['unique_presentations']}")
    if hybrid_metadata['multi_slot_presentations'] > 0:
        print(f"  ({hybrid_metadata['multi_slot_presentations']} presentations span multiple slots)")
    
    print("\nHybrid Sessions:")
    print(df_hybrid_sessions[["session_id", "title", "presentation_count"]].to_string(index=False))
else:
    print("No hybrid sessions file - proceeding without pre-assigned sessions")

✓ Loaded 34 hybrid presentation slots
✓ Found 6 hybrid sessions
✓ Unique presentations: 26
  (6 presentations span multiple slots)

Hybrid Sessions:
session_id                                                                       title  presentation_count
HYBRID-001   Advancing Circular Bioeconomy Systems (CBS): Opportunities and Challenges                   4
HYBRID-002         Autonomous Machine Safety: Emerging Risks and Protective Strategies                   7
HYBRID-003                  Heat and Environmental Exposures: Impacts on Worker Safety                   5
HYBRID-004                                                       Irrigation Management                   4
HYBRID-005 Remote Estimation of Evapotranspiration in Natural and Agricultural Systems                   7
HYBRID-006                               Failed Research: Challenges and Opportunities                   7


---
## 5. Generate Embeddings for Hybrid Presentations

In [6]:
hybrid_embeddings = None
hybrid_abstract_ids = None

if df_hybrid is not None:
    # Generate embeddings for hybrid presentations
    hybrid_texts = df_hybrid["combined_text"].tolist()
    print(f"Generating embeddings for {len(hybrid_texts)} hybrid presentations...")
    
    hybrid_embeddings_list = embedder.embed_batch(hybrid_texts, show_progress=True)
    hybrid_embeddings = np.array(hybrid_embeddings_list)
    hybrid_abstract_ids = df_hybrid["abstract_id"].tolist()
    
    print(f"✓ Generated embeddings: shape {hybrid_embeddings.shape}")
    
    # Save embeddings to database
    conference_db.import_presentations_batch(df_hybrid.to_dict('records'))
    print(f"✓ Imported hybrid presentations to database")

Generating embeddings for 34 hybrid presentations...
Found 0/34 embeddings in cache
Generating 34 new embeddings...
✓ Generated embeddings: shape (34, 3072)
✓ Imported hybrid presentations to database


---
## 6. Load Main Presentations

In [7]:
# Inspect main presentations file
info = inspect_file(PRESENTATIONS_FILE)
print(f"Presentations file: {info['row_count']} rows, {info['column_count']} columns")
print("\nAuto-detected mappings:")
for field, col in info['auto_detected_mappings'].items():
    print(f"  {field}: {col}")

Presentations file: 1129 rows, 56 columns

Auto-detected mappings:
  title: Submission-Call for Abstracts-Presentation Title-Character max 160
  abstract: Submission-Call for Abstracts-Submission ID - 7 digits
  submission_id: Submission ID
  presenter_first_name: Owner-First Name
  presenter_last_name: Owner-Last Name
  presenter_email: Owner-E-mail Address
  affiliation: Owner-Company/University
  technical_community: Submission-Call for Abstracts-Technical Community-First Preference 
  session_preference: Submission-Call for Abstracts-Select Your Session Preference


In [8]:
# Create column mapping
presentations_mapping = ColumnMapping(
    title=PRESENTATIONS_COLUMNS["title"],
    abstract=PRESENTATIONS_COLUMNS["abstract"],
    submission_id=PRESENTATIONS_COLUMNS["submission_id"],
    session_preference=PRESENTATIONS_COLUMNS.get("session_preference"),
    technical_community=PRESENTATIONS_COLUMNS.get("technical_community"),
    presenter_first_name=PRESENTATIONS_COLUMNS.get("presenter_first_name"),
    presenter_last_name=PRESENTATIONS_COLUMNS.get("presenter_last_name"),
    presenter_email=PRESENTATIONS_COLUMNS.get("presenter_email"),
)

# Load presentations
df_presentations, pres_metadata = load_presentations(
    PRESENTATIONS_FILE,
    presentations_mapping,
    conference_year=CONFERENCE_YEAR,
)

print(f"✓ Loaded {pres_metadata['loaded_count']} presentations")
print(f"  - Real IDs: {pres_metadata['real_id_count']}")
print(f"  - Temp IDs: {pres_metadata['temp_id_count']}")

✓ Loaded 1129 presentations
  - Real IDs: 1129
  - Temp IDs: 0


In [9]:
# Filter to oral presentations only
if "session_preference" in df_presentations.columns and ORAL_PREFERENCE_TEXT:
    df_oral = df_presentations[df_presentations["session_preference"] == ORAL_PREFERENCE_TEXT].copy()
    print(f"✓ Filtered to {len(df_oral)} oral presentations")
else:
    df_oral = df_presentations.copy()
    print(f"Using all {len(df_oral)} presentations (no session preference filter)")

✓ Filtered to 1129 oral presentations


---
## 7. Generate Embeddings for Main Presentations

In [10]:
# Generate embeddings for oral presentations
oral_texts = df_oral["combined_text"].tolist()
print(f"Generating embeddings for {len(oral_texts)} presentations...")

oral_embeddings_list = embedder.embed_batch(oral_texts, show_progress=True)
oral_embeddings = np.array(oral_embeddings_list)
oral_abstract_ids = df_oral["abstract_id"].tolist()

print(f"✓ Generated embeddings: shape {oral_embeddings.shape}")

Generating embeddings for 1129 presentations...
Found 0/1129 embeddings in cache
Generating 1129 new embeddings...
✓ Generated embeddings: shape (1129, 3072)


---
## 8. Remove Near-Duplicates

In [11]:
# Calculate similarity matrix for duplicate detection
similarity_matrix = cosine_similarity(oral_embeddings, oral_embeddings)

# Find near-duplicates
duplicates_to_remove = []
n = len(oral_abstract_ids)

for i in range(n):
    for j in range(i + 1, n):
        if similarity_matrix[i, j] >= DUPLICATE_THRESHOLD:
            # Keep the one with lower index (earlier submission)
            print(f"Near-duplicate: {oral_abstract_ids[i]} ↔ {oral_abstract_ids[j]} (sim={similarity_matrix[i, j]:.4f})")
            duplicates_to_remove.append(j)

# Remove duplicates
duplicates_to_remove = sorted(set(duplicates_to_remove))
print(f"\nFound {len(duplicates_to_remove)} near-duplicate presentations to remove")

if duplicates_to_remove:
    # Create mask for keeping
    keep_mask = [i not in duplicates_to_remove for i in range(n)]
    
    # Filter data
    df_oral = df_oral.iloc[keep_mask].reset_index(drop=True)
    oral_embeddings = oral_embeddings[keep_mask]
    oral_abstract_ids = [aid for i, aid in enumerate(oral_abstract_ids) if keep_mask[i]]
    
    print(f"✓ Remaining presentations: {len(df_oral)}")

Near-duplicate: 2600294 ↔ 2600295 (sim=0.9932)
Near-duplicate: 2600627 ↔ 2600624 (sim=0.9908)

Found 2 near-duplicate presentations to remove
✓ Remaining presentations: 1127


---
## 9. Create Sessions with Hierarchical Clustering

In [12]:
# Build hybrid assignments dict (abstract_id -> session_id)
hybrid_assignments = {}
if df_hybrid is not None:
    for _, row in df_hybrid.iterrows():
        hybrid_assignments[row["abstract_id"]] = row["session_id"]

print(f"Hybrid assignments: {len(hybrid_assignments)} presentations pre-assigned")

# Create placement strategy
if hybrid_assignments:
    # Use HybridFirstPlacement to fill hybrid sessions first
    strategy = create_placement_strategy("hybrid_first")
    print(f"Using strategy: {strategy.name}")
else:
    # Use standard OralSessionPlacement
    strategy = create_placement_strategy("oral")
    print(f"Using strategy: {strategy.name}")

Hybrid assignments: 34 presentations pre-assigned
Using strategy: hybrid_first_hierarchical


In [13]:
# Set session constraints
constraints = SessionConstraints(
    min_session_size=MIN_SESSION_SIZE,
    max_session_size=MAX_SESSION_SIZE,
    max_sessions=MAX_SESSIONS,
)

# Prepare placement arguments
place_kwargs = {
    "embeddings": oral_embeddings,
    "abstract_ids": oral_abstract_ids,
    "constraints": constraints,
    "hybrid_assignments": hybrid_assignments if hybrid_assignments else None,
}

# Add hybrid embeddings if using hybrid_first strategy
if hybrid_assignments and hybrid_embeddings is not None:
    place_kwargs["hybrid_embeddings"] = dict(zip(hybrid_abstract_ids, hybrid_embeddings))

# Run placement
print("Creating sessions...")
result = strategy.place(**place_kwargs)

print(f"\n✓ Created {len(result.sessions)} sessions")
print(f"  - Assigned: {len(result.session_assignments)} presentations")
print(f"  - Unassigned: {len(result.unassigned)} presentations")

Creating sessions...

✓ Created 106 sessions
  - Assigned: 1161 presentations
  - Unassigned: 0 presentations


In [14]:
# Add session assignments to DataFrame
df_oral["session_id"] = df_oral["abstract_id"].map(result.session_assignments)

# Import presentations to database
conference_db.import_presentations_batch(df_oral.to_dict('records'))

# Save sessions to database
for session in result.sessions:
    conference_db.create_session(
        session_id=session["session_id"],
        presentation_ids=session["presentation_ids"],
        is_hybrid=session.get("is_hybrid", False),
        placement_strategy=strategy.name,
    )

print(f"✓ Saved {len(result.sessions)} sessions to database")

✓ Saved 106 sessions to database


---
## 10. Calculate Session Metrics

In [15]:
# Calculate all metrics
metrics = calculate_all_metrics(
    oral_embeddings,
    oral_abstract_ids,
    result.session_assignments,
)

print(f"Session Metrics Summary:")
print(f"  - Mean Coherence: {metrics['summary']['mean_coherence']:.3f}")
print(f"  - Std Coherence: {metrics['summary']['std_coherence']:.3f}")
print(f"  - Mean Fit: {metrics['summary']['mean_fit']:.3f}")

# Update session metrics in database
for session_id, sm in metrics["session_metrics"].items():
    conference_db.update_session_metrics(
        session_id, 
        sm["coherence"], 
        sm.get("distinctiveness")
    )

Session Metrics Summary:
  - Mean Coherence: 0.863
  - Std Coherence: 0.027
  - Mean Fit: 0.859


In [16]:
# Create sessions DataFrame for further processing
sessions_data = []
for session_id, sm in metrics["session_metrics"].items():
    sessions_data.append({
        "session_id": session_id,
        "size": sm["size"],
        "coherence": sm["coherence"],
        "distinctiveness": sm.get("distinctiveness", 0),
        "presentation_ids": sm["presentation_ids"],
    })

df_sessions = pd.DataFrame(sessions_data)
df_sessions = df_sessions.sort_values("session_id").reset_index(drop=True)

print(f"\nSessions Summary:")
print(df_sessions[["session_id", "size", "coherence", "distinctiveness"]].to_string(index=False))


Sessions Summary:
 session_id  size  coherence  distinctiveness
 HYBRID-001     5   0.861784         0.024484
 HYBRID-002     2   0.929818         0.053191
 HYBRID-003     4   0.850437         0.030321
 HYBRID-004     5   0.882568         0.029812
 HYBRID-005     2   0.918613         0.030505
 HYBRID-006     2   0.853432         0.068416
SESSION-001     9   0.903183         0.021796
SESSION-002    10   0.904577         0.021405
SESSION-003    10   0.903036         0.018824
SESSION-004    12   0.900334         0.016780
SESSION-005    10   0.901656         0.024227
SESSION-006     9   0.902468         0.023179
SESSION-007    11   0.894547         0.029991
SESSION-008    11   0.893957         0.027259
SESSION-009    10   0.897470         0.024533
SESSION-010    15   0.892747         0.018824
SESSION-011     9   0.889708         0.031075
SESSION-012    10   0.888811         0.028665
SESSION-013    13   0.889017         0.027465
SESSION-014    11   0.883843         0.026708
SESSION-015    

---
## 11. Generate Session Titles and Keywords

In [17]:
# Initialize title generator
title_generator = GeminiTitleGenerator(model=TITLE_MODEL)
print(f"✓ Title generator initialized: {TITLE_MODEL}")

✓ Title generator initialized: gemini-2.5-flash-lite


In [18]:
# Generate titles for each session
titles_list = []
keywords_list = []

for idx, row in df_sessions.iterrows():
    session_id = row["session_id"]
    pres_ids = row["presentation_ids"]
    
    # Get presentation details
    session_presentations = df_oral[df_oral["abstract_id"].isin(pres_ids)]
    pres_list = [
        {"title": r["title"], "abstract": r.get("abstract", "")}
        for _, r in session_presentations.iterrows()
    ]
    
    print(f"Generating title for {session_id} ({len(pres_list)} presentations)...")
    
    try:
        result = title_generator.generate(pres_list)
        titles_list.append(result.titles)
        keywords_list.append(result.keywords)
        
        # Update database
        conference_db.update_session_titles(
            session_id,
            result.titles,
            result.keywords,
            model_name=TITLE_MODEL,
        )
        
        print(f"  → {result.titles[0] if result.titles else 'No title'}")
    except Exception as e:
        print(f"  Error: {e}")
        titles_list.append([])
        keywords_list.append([])

# Add to sessions DataFrame
df_sessions["title"] = [t[0] if t else "" for t in titles_list]
df_sessions["title_options"] = titles_list
df_sessions["keywords"] = [", ".join(k) for k in keywords_list]

Generating title for HYBRID-001 (5 presentations)...
  → Valorizing Biomass Waste: From Feedstocks to Bioenergy & Bioproducts
Generating title for HYBRID-002 (2 presentations)...
  → Autonomous Ag Machinery: Bridging Innovation and Safety
Generating title for HYBRID-003 (4 presentations)...
  → Enhancing Safety and Comfort in Agricultural Environments
Generating title for HYBRID-004 (5 presentations)...
  → Precision Irrigation: Optimizing Water Use for Resilient Agriculture
Generating title for HYBRID-005 (2 presentations)...
  → Advancing Water Use Estimation with Remote Sensing & ML
Generating title for HYBRID-006 (2 presentations)...
  → Engineering Solutions for Nutrient Management: From Research to Reality
Generating title for SESSION-001 (9 presentations)...
  → AI-Powered Precision Livestock: Vision for Welfare & Management
Generating title for SESSION-002 (10 presentations)...
  → Intelligent Robotics for Precision Weed Management
Generating title for SESSION-003 (10 presentat

In [19]:
# Display sessions with titles
print("\nGenerated Session Titles:")
print(df_sessions[["session_id", "title", "keywords", "size"]].to_string(index=False))


Generated Session Titles:
 session_id                                                                                       title                                                                                                                                                                                                                keywords  size
 HYBRID-001                        Valorizing Biomass Waste: From Feedstocks to Bioenergy & Bioproducts                                                                                                                             biomass valorization, biorefining, circular economy, technoeconomics, feedstock flexibility     5
 HYBRID-002                                     Autonomous Ag Machinery: Bridging Innovation and Safety                                                                                                                autonomous machinery, agricultural safety, human-robot interaction, risk assessment, technology adoption  

---
## 12. Match Sessions to Committees (Optional)

In [20]:
df_committees = None
committee_embeddings = None

if Path(COMMITTEES_FILE).exists():
    # Load committees
    df_committees, committee_metadata = load_committees(
        COMMITTEES_FILE,
        name_column="Committee_Name",
        description_column="Description",
    )
    
    print(f"✓ Loaded {committee_metadata['loaded_count']} committees")
    
    # Generate embeddings for committees
    committee_texts = df_committees["combined_text"].tolist()
    committee_embeddings_list = embedder.embed_batch(committee_texts, show_progress=True)
    committee_embeddings = np.array(committee_embeddings_list)
    
    print(f"✓ Committee embeddings: shape {committee_embeddings.shape}")
else:
    print(f"No committees file found: {COMMITTEES_FILE}")
    print("Skipping committee matching...")

✓ Loaded 108 committees
Found 0/108 embeddings in cache
Generating 108 new embeddings...
✓ Committee embeddings: shape (108, 3072)


In [21]:
if df_committees is not None and committee_embeddings is not None:
    # Match sessions to committees
    # Calculate session centroids
    session_matches = []
    
    for idx, row in df_sessions.iterrows():
        session_id = row["session_id"]
        pres_ids = row["presentation_ids"]
        
        # Get embeddings for session presentations
        pres_indices = [oral_abstract_ids.index(pid) for pid in pres_ids if pid in oral_abstract_ids]
        if not pres_indices:
            continue
            
        session_embs = oral_embeddings[pres_indices]
        session_centroid = np.mean(session_embs, axis=0, keepdims=True)
        
        # Calculate similarity to committees
        similarities = cosine_similarity(session_centroid, committee_embeddings)[0]
        
        # Get top 3 matches
        top_indices = np.argsort(similarities)[-3:][::-1]
        
        matches = []
        for rank, idx in enumerate(top_indices, 1):
            committee_id = df_committees.iloc[idx]["committee_id"]
            committee_name = df_committees.iloc[idx]["committee_name"]
            score = similarities[idx]
            matches.append((committee_id, committee_name, score, rank))
            
            # Store in database
            conference_db.store_session_committee_matches(
                session_id,
                [(committee_id, float(score), rank)]
            )
        
        session_matches.append({
            "session_id": session_id,
            "committee_1": matches[0][1] if matches else "",
            "score_1": matches[0][2] if matches else 0,
            "committee_2": matches[1][1] if len(matches) > 1 else "",
            "score_2": matches[1][2] if len(matches) > 1 else 0,
        })
    
    # Add committee matches to sessions
    df_committee_matches = pd.DataFrame(session_matches)
    df_sessions = df_sessions.merge(df_committee_matches, on="session_id", how="left")
    
    print("\nCommittee Assignments:")
    print(df_sessions[["session_id", "title", "committee_1", "score_1"]].head(10).to_string(index=False))


Committee Assignments:
 session_id                                                                   title                                            committee_1  score_1
 HYBRID-001    Valorizing Biomass Waste: From Feedstocks to Bioenergy & Bioproducts           CBSI - Circular Bioeconomy Systems Institute 0.897709
 HYBRID-002                 Autonomous Ag Machinery: Bridging Innovation and Safety ESH-04/2 Farmers With Disabilities Technology Exchange 0.859619
 HYBRID-003               Enhancing Safety and Comfort in Agricultural Environments ASE-347 and US TAG TC 347 Data-driven agrifood systems 0.871007
 HYBRID-004    Precision Irrigation: Optimizing Water Use for Resilient Agriculture                         NRES-244 Irrigation Management 0.898477
 HYBRID-005                 Advancing Water Use Estimation with Remote Sensing & ML                         NRES-244 Irrigation Management 0.846851
 HYBRID-006 Engineering Solutions for Nutrient Management: From Research to Reality     

---
## 13. Export Results

In [22]:
# Export presentations with session assignments
df_export = conference_db.export_to_dataframe(flatten_extra_data=True)

# Get session titles for presentations
session_titles = dict(zip(df_sessions["session_id"], df_sessions["title"]))
df_export["session_title"] = df_export["session_id"].map(session_titles)

print(f"Export DataFrame: {len(df_export)} rows, {len(df_export.columns)} columns")
print(f"Columns: {list(df_export.columns)}")

Export DataFrame: 1161 rows, 65 columns
Columns: ['abstract_id', 'is_invited', 'source_submission_id', 'source_abstract_id', 'slot_number', 'title', 'abstract', 'presenter_first_name', 'presenter_last_name', 'presenter_email', 'affiliation', 'technical_community', 'session_preference', 'Session ID', 'Submission ID', 'Submission Created Date & Time', 'External reference', 'Submission Completed Date & Time', 'Submission Status', 'Acceptance Status', '# Reviews', 'Rating', 'Std Dev', 'Owner-Company/University', 'Owner-City', 'Owner-State', 'Owner-Country', 'Owner-CC Email', 'Owner-Test Profile', 'Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement', 'Submission-Call for Abstracts-Anticipated Session Topics', 'Submission-Call for Abstracts-Technical Community-Second Preference ', 'Submission-Call for Abstracts-Session', 'Submission-Call for Abstracts-Order value (for HQ use)', 'Submission-Presentation Edits-Would like to be moved 

In [23]:
# Save exports
output_prefix = f"{OUTPUT_DIR}/{CONFERENCE_NAME}"

# Full presentations export
df_export.to_parquet(f"{output_prefix}_presentations.parquet", compression='snappy')
print(f"✓ Saved: {output_prefix}_presentations.parquet")

# Sessions export
df_sessions.to_parquet(f"{output_prefix}_sessions.parquet", compression='snappy')
df_sessions.to_csv(f"{output_prefix}_sessions.csv", index=False)
print(f"✓ Saved: {output_prefix}_sessions.parquet")
print(f"✓ Saved: {output_prefix}_sessions.csv")

# Similarity matrices for viewer
pres_sim_matrix = cosine_similarity(oral_embeddings, oral_embeddings)
df_pres_sim = pd.DataFrame(pres_sim_matrix, index=oral_abstract_ids, columns=oral_abstract_ids)
df_pres_sim.to_parquet(f"{output_prefix}_pres_similarities.parquet", compression='snappy')
print(f"✓ Saved: {output_prefix}_pres_similarities.parquet")

print(f"\n✓ All exports complete!")

✓ Saved: output/AIM2026-TestNB_presentations.parquet
✓ Saved: output/AIM2026-TestNB_sessions.parquet
✓ Saved: output/AIM2026-TestNB_sessions.csv
✓ Saved: output/AIM2026-TestNB_pres_similarities.parquet

✓ All exports complete!


In [24]:
# Create redacted export (no PII) for web viewer
columns_to_redact = [
    "presenter_email", "presenter_first_name", "presenter_last_name",
    "affiliation", "abstract"
]
df_redacted = df_export.drop(columns=[c for c in columns_to_redact if c in df_export.columns], errors='ignore')
df_redacted.to_parquet(f"{output_prefix}_presentations_public.parquet", compression='snappy')
print(f"✓ Saved redacted export: {output_prefix}_presentations_public.parquet")

✓ Saved redacted export: output/AIM2026-TestNB_presentations_public.parquet


In [ ]:
# Export Viewer Bundle (recommended for session_viewer_app.py)
from smart.io.exporters import export_viewer_bundle

bundle_path = f"{OUTPUT_DIR}/{CONFERENCE_NAME}_bundle"

manifest = export_viewer_bundle(
    df_presentations=df_export,
    df_sessions=df_sessions,
    embeddings=oral_embeddings,
    abstract_ids=oral_abstract_ids,
    output_path=bundle_path,
    conference_name=CONFERENCE_NAME,
    include_abstracts=False,  # Set True to include abstracts in public export
    include_embeddings=True,  # Include for future metric recalculation
    encrypt_password=None,    # Set password to encrypt PII data
)

print(f"✓ Viewer bundle exported to: {bundle_path}")
print(f"  - Presentations: {manifest['presentation_count']}")
print(f"  - Sessions: {manifest['session_count']}")
print(f"  - Has embeddings: {manifest['has_embeddings']}")
print(f"\nTo view results, run:")
print(f"  streamlit run session_viewer_app.py -- --bundle {bundle_path}")

---
## Summary

In [25]:
# Final summary
print("="*60)
print("SMART SESSION ORGANIZATION COMPLETE")
print("="*60)
print(f"\nConference: {CONFERENCE_NAME}")
print(f"\nPresentations:")
print(f"  - Total loaded: {len(df_oral)}")
print(f"  - Assigned to sessions: {len(result.session_assignments)}")
print(f"  - Unassigned: {len(result.unassigned)}")
print(f"\nSessions:")
print(f"  - Total sessions: {len(df_sessions)}")
if df_hybrid_sessions is not None:
    print(f"  - Hybrid sessions: {len(df_hybrid_sessions)}")
print(f"  - Mean coherence: {metrics['summary']['mean_coherence']:.3f}")
print(f"  - Mean fit: {metrics['summary']['mean_fit']:.3f}")
print(f"\nOutput files saved to: {OUTPUT_DIR}/")
print(f"\nCache stats:")
cache_stats = embedding_cache.get_stats()
print(f"  - Total embeddings cached: {cache_stats['total_embeddings']}")

SMART SESSION ORGANIZATION COMPLETE

Conference: AIM2026-TestNB

Presentations:
  - Total loaded: 1127


AttributeError: 'TitleGenerationResult' object has no attribute 'session_assignments'